# Model Building - SVD and Popularity Penalization

In [1]:
import pandas as pd
from surprise import Dataset, Reader

# load ratings and sample 2,000,000 rows
ratings = pd.read_csv('../../ml-25m/ratings.csv')
sample = ratings.sample(n=2_000_000, random_state=42)

# load into Surprise
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(sample[['userId', 'movieId', 'rating']], reader)

print('Sample shape:', sample.shape)

Sample shape: (2000000, 4)


In [2]:
movies = pd.read_csv('../../ml-25m/movies.csv')

# build film_popularity from the sample
film_popularity = (
    sample.groupby('movieId')
    .agg(rating_count=('rating', 'count'), avg_rating=('rating', 'mean'))
    .reset_index()
    .merge(movies[['movieId', 'title']], on='movieId', how='left')
    [['movieId', 'title', 'rating_count', 'avg_rating']]
    .sort_values('rating_count', ascending=False)
    .reset_index(drop=True)
)
film_popularity['avg_rating'] = film_popularity['avg_rating'].round(3)


def popularity_baseline(film_popularity, n=10):
    """Return the top N films by rating_count."""
    return film_popularity.nlargest(n, 'rating_count')[['title', 'rating_count', 'avg_rating']].reset_index(drop=True)


top_films = popularity_baseline(film_popularity)
print(top_films)

                                       title  rating_count  avg_rating
0                        Forrest Gump (1994)          6524       4.035
1                        Pulp Fiction (1994)          6477       4.186
2           Shawshank Redemption, The (1994)          6451       4.423
3                         Matrix, The (1999)          5865       4.160
4           Silence of the Lambs, The (1991)          5791       4.162
5  Star Wars: Episode IV - A New Hope (1977)          5545       4.134
6                       Jurassic Park (1993)          5140       3.673
7                    Schindler's List (1993)          4828       4.250
8                          Braveheart (1995)          4784       3.994
9                          Fight Club (1999)          4701       4.242


In [3]:
from surprise import SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd = SVD(n_factors=100, random_state=42)
svd.fit(trainset)

predictions = svd.test(testset)
rmse = accuracy.rmse(predictions)
mae = accuracy.mae(predictions)

RMSE: 0.9048
MAE:  0.6946


In [4]:
import random

def get_svd_recommendations(user_id, svd, all_movie_ids, rated_movie_ids, n=10):
    """Predict scores for all unrated films and return the top N movie IDs."""
    unrated = [mid for mid in all_movie_ids if mid not in rated_movie_ids]
    predictions = [svd.predict(user_id, mid) for mid in unrated]
    predictions.sort(key=lambda x: x.est, reverse=True)
    return [pred.iid for pred in predictions[:n]]


all_movie_ids = set(sample['movieId'].unique())
user_rated = sample.groupby('userId')['movieId'].apply(set).to_dict()

all_user_ids = list(user_rated.keys())
random.seed(42)
sample_users = random.sample(all_user_ids, 500)

user_recs = {}
for user_id in sample_users:
    user_recs[user_id] = get_svd_recommendations(
        user_id, svd, all_movie_ids, user_rated[user_id]
    )

print(f'Users processed: {len(user_recs)}')

Users processed: 500


In [5]:
import numpy as np

# build log-normalized popularity scores in [0, 1]
log_counts = np.log1p(film_popularity.set_index('movieId')['rating_count'])
pop_dict = (log_counts / log_counts.max()).to_dict()


def get_penalized_recommendations(user_id, svd, all_movie_ids, rated_movie_ids, n=10, lambda_penalty=0.3):
    """Return top N movie IDs after penalizing CF scores by popularity."""
    unrated = [mid for mid in all_movie_ids if mid not in rated_movie_ids]
    predictions = [svd.predict(user_id, mid) for mid in unrated]
    adjusted = [
        (pred.iid, pred.est * (1 - lambda_penalty * pop_dict.get(pred.iid, 0.0)))
        for pred in predictions
    ]
    adjusted.sort(key=lambda x: x[1], reverse=True)
    return [iid for iid, _ in adjusted[:n]]


penalized_recs = {}
for user_id in sample_users:
    penalized_recs[user_id] = get_penalized_recommendations(
        user_id, svd, all_movie_ids, user_rated[user_id], lambda_penalty=0.3
    )

print(f'Users processed: {len(penalized_recs)}')

Users processed: 500


In [ ]:
import joblib, os

# persist the trained SVD model so downstream notebooks (05_sparsity_experiment)
# and any future inference scripts can reload it without retraining.
os.makedirs('../outputs', exist_ok=True)
joblib.dump(svd, '../outputs/svd_model.pkl')
print('SVD model saved  →  outputs/svd_model.pkl')

In [ ]:
# experiment A with hybrid model (α=0.6) across sparsity levels.
# results are precomputed by run_hybrid_sparsity.py (takes hours to run).

import pandas as pd, os

results_path = '../outputs/sparsity_expA_all_models.csv'

if os.path.exists(results_path):
    sparsity_hybrid_df = pd.read_csv(results_path)
    print(sparsity_hybrid_df.to_string(index=False))
else:
    print(f'Results not yet computed. Run run_hybrid_sparsity.py first.')
    print('  python run_hybrid_sparsity.py')


In [ ]:
# experiment B: per-movie subsampling (fixed catalogue). Combined A+B results
# from run_hybrid_sparsity_B.py.

import pandas as pd, os

results_path = '../outputs/sparsity_expAB_combined.csv'

if os.path.exists(results_path):
    sparsity_results = pd.read_csv(results_path)
    print(f"Loaded {len(sparsity_results)} rows  ({sparsity_results['experiment'].nunique()} experiments)")
    print()
    for exp in sorted(sparsity_results['experiment'].unique()):
        sub = sparsity_results[sparsity_results['experiment'] == exp]
        print(f"=== Experiment {exp} ===")
        print(sub[['sparsity','n_ratings','model','ndcg','ci_lower','ci_upper']].to_string(index=False))
        print()
else:
    print(f'Combined results not yet available at {results_path}')
    print('Run run_hybrid_sparsity.py (Exp A) and run_hybrid_sparsity_B.py (Exp B) first.')
